In [ ]:
# 1. ScienceQA
#   with the train_sft_pt script, when using ScienceQA
from transformers import AutoTokenizer, AutoModelForCausalLM
from data.pt_dataset import ScienceQADataset
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Loaded 4241 samples
--- Prompt ---
Question: Which figure of speech is used in this text?
Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans.
—Homer, The Iliad
A) chiasmus
B) apostrophe
Answer:

--- Generation ---
 A) chiasmus

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question:

--- Reference ---
 Figures of speech are words or phrases that use language in a nonliteral or unusual way. They can make writing more expressive.
Anaphora is the repetition of the same word or words at the beginning of several phrases or clauses.
We are united. We are po

### 1. Question Answering | ScienceQA

In [8]:
# 1. ScienceQA
# - the evaluation pipeline looks fine to me, but I seems to encouter error 
#   with the train_sft_pt script, when using ScienceQA
from data.pt_dataset import ScienceQADataset

dataset = ScienceQADataset(split="test", tokenizer=tokenizer, max_length=512)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len]))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len], 
    max_new_tokens=128, 
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(full_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference ---")
print(ref_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred: {pred_answer}")
print(f"Extracted Gold: {gold_answer}")
print(f"Correct: {pred_answer == gold_answer if gold_answer else False}")

Loaded 4241 samples
--- Prompt ---
Question: Which figure of speech is used in this text?
Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans.
—Homer, The Iliad
A) chiasmus
B) apostrophe
Answer:

--- Generation ---
 A) chiasmus

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question:

--- Reference ---
 Figures of speech are words or phrases that use language in a nonliteral or unusual way. They can make writing more expressive.
Anaphora is the repetition of the same word or words at the beginning of several phrases or clauses.
We are united. We are po

#### ARC

In [ ]:
from data.pt_dataset import ARCDataset


arc_ds = ARCDataset(split="test", tokenizer=tokenizer, max_length=256)
print(f"Loaded {len(arc_ds)} samples")

arc_ds = ARCDataset(split="test", tokenizer=tokenizer, max_length=256)
print(f"Loaded {len(arc_ds)} samples")

# --- model generation (rollout)
idx = 0
sample = arc_ds[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len]))

print("\n--- Generation ---")
with torch.no_grad():
    generated = model.generate(
        input_ids[:, :prompt_len],
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(full_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):])

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference ---")
print(ref_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):])

pred_answer = arc_ds.extract_answer(full_text)
gold_answer = arc_ds.extract_answer(ref_text)

print(f"\nExtracted Pred: {pred_answer}")
print(f"Extracted Gold: {gold_answer}")
print(f"Correct: {pred_answer == gold_answer if gold_answer else False}")

### 2. Code Evaluation (MBPP / HumanEval / LiveCodeBench)

In [ ]:
import torch
from data.pt_dataset import (
    get_dataset, MBPPDataset, HumanEvalDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
# he_ds  = get_dataset("humaneval",     split="test", tokenizer=tokenizer, max_length=1024)
# lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP: {len(mbpp_ds)} problems")

# --- ground-truth sanity check
ex0 = mbpp_ds.dataset[0]
gt_result = check_code_correctness(ex0["code"], mbpp_ds.get_test_cases(0), timeout=10)
print(f"Ground-truth: {'✅' if gt_result['passed'] else '❌'}  ({gt_result['num_passed']}/{gt_result['num_total']} tests)")

# --- model generation (rollout)
ds = mbpp_ds
idx = 0
sample = ds[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

prompt_text = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)
print("\n--- Prompt ---")
print(prompt_text)

with torch.no_grad():
    gen_ids = model.generate(
        input_ids[:, :prompt_len],
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
generated_part = full_text[len(prompt_text):]
print("\n--- Generated ---")
print(generated_part)

# --- execution-based evaluation
tests = ds.get_test_cases(idx)
result = check_code_correctness(full_text, tests, timeout=10)

print(f"\nTests: {tests}")
print(f"Result: {'✅ PASS' if result['passed'] else '❌ FAIL'}  ({result['num_passed']}/{result['num_total']})")

MBPP      : 257 problems

MBPP[0] ground-truth: ✅  (3/3 tests)


### Function Calling

In [ ]:
import importlib, data.pt_dataset as _pt
from data.pt_dataset import XLAMDataset
import json

dataset = XLAMDataset(split="test", tokenizer=tokenizer, max_length=1024)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len],
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
generated_part = full_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(generated_part)

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference (gold response) ---")
ref_part = ref_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(ref_part)

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred : {pred_answer}")
print(f"Extracted Gold : {gold_answer}")
print(f"Correct        : {pred_answer == gold_answer if gold_answer else False}")

# --- Pretty-print parsed JSON for readability
if pred_answer:
    try:
        print(f"\nParsed pred:\n{json.dumps(json.loads(pred_answer), indent=2)}")
    except Exception:
        pass
if gold_answer:
    try:
        print(f"\nParsed gold:\n{json.dumps(json.loads(gold_answer), indent=2)}")
    except Exception:
        pass

### More Datasets

In the notebook, we need to check the "length" for each dataset (max_len & max_generation_len)

1. BoolQ | 3.2k test, >9k train (fine) | this one got "passage" which is very long (max length 2.5k should be fine here), "MAX_LENGTH ~2.5K", response length very short though, there is no CoT, but we can add <abs> before the answer token id

2. OpenBookQA | "main" & "additional", each has ~5k train, 500 val, 500 test (combine val & test into test, combine "main and additional", so total 10k train, 2k test, evaluation number set to 2k) | query < 330, response < 150, choices, so max_length 768 is safer, although the response length is not clear, better have a kernel to verify

3. AQuA, question < 400, answer < 740, max_len=1024, max_response_len=768 suitable | train 5k, test 500 (bit small test set, room for variations)

4. hotpotqa/hotpot_qa | is it possible to include this (with a special "answer_token_id"?)
   cap to 20k train & 4k test
   


In [ ]:
"""
Reusable length & format inspector for any dataset class.
Prints: split sizes, token length distribution (prompt, response, total),
        sample prompt/response text, extract_answer check, and abs-prefix insertion demo.
"""
import numpy as np
from data.pt_dataset import (
    BoolQDataset, OpenBookQADataset, AQuADataset, CommonsenseQADataset, MMLUDataset,
    get_answer_start_index,
)
from sorl.sorl_trainer import insert_prefix_abs
import torch

def inspect_dataset(DatasetCls, tokenizer, max_length, split="train",
                    n_samples_text=3, n_abs=8, placeholder_token=151936,
                    pad_token_id=None):
    """Load dataset, compute length stats, show samples, test extract_answer & abs prefix."""
    pad_id = pad_token_id or tokenizer.pad_token_id or tokenizer.eos_token_id

    ds = DatasetCls(split=split, tokenizer=tokenizer, max_length=max_length)
    n = len(ds)
    print(f"{'='*70}")
    print(f"  {DatasetCls.__name__}  split={split}  max_length={max_length}  n={n}")
    print(f"{'='*70}")

    # --- length stats (sample up to 2000 for speed) ---
    sample_n = min(n, 2000)
    prompt_lens, response_lens, total_lens = [], [], []
    for i in range(sample_n):
        s = ds[i]
        attn = s["attention_mask"]
        total = int(attn.sum().item())
        pl = int(s["prompt_len"])
        prompt_lens.append(pl)
        response_lens.append(total - pl)
        total_lens.append(total)

    for name, vals in [("prompt", prompt_lens), ("response", response_lens), ("total", total_lens)]:
        a = np.array(vals)
        print(f"  {name:>10s}  min={a.min():5d}  median={int(np.median(a)):5d}  "
              f"mean={a.mean():7.1f}  p95={int(np.percentile(a,95)):5d}  max={a.max():5d}")

    truncated = sum(1 for t in total_lens if t >= max_length)
    print(f"  truncated at max_length: {truncated}/{sample_n} ({100*truncated/sample_n:.1f}%)")
    print()

    # --- show sample texts ---
    for i in range(min(n_samples_text, n)):
        s = ds[i]
        ids = s["input_ids"]
        pl = int(s["prompt_len"])
        attn = s["attention_mask"]
        valid = int(attn.sum().item())

        prompt_text = tokenizer.decode(ids[:pl], skip_special_tokens=True)
        response_text = tokenizer.decode(ids[pl:valid], skip_special_tokens=True)
        full_text = tokenizer.decode(ids[:valid], skip_special_tokens=True)

        print(f"  --- Sample {i} (prompt={pl} tok, response={valid-pl} tok) ---")
        print(f"  PROMPT: {prompt_text[:200]}{'...' if len(prompt_text)>200 else ''}")
        print(f"  RESPONSE: {response_text[:200]}{'...' if len(response_text)>200 else ''}")

        # extract_answer
        gold = ds.extract_answer(full_text)
        print(f"  EXTRACTED ANSWER: {gold}")
        print()

    # --- abs prefix insertion demo (first sample) ---
    s0 = ds[0]
    ids = s0["input_ids"].unsqueeze(0)
    attn = s0["attention_mask"].unsqueeze(0)
    pl = torch.tensor([int(s0["prompt_len"])])

    new_ids, new_attn = insert_prefix_abs(ids, attn, pl, n_abs, placeholder_token, pad_id)
    valid_new = int(new_attn[0].sum().item())
    abs_positions = (new_ids[0] == placeholder_token).nonzero(as_tuple=True)[0].tolist()

    print(f"  --- ABS prefix insertion (n_abs={n_abs}) ---")
    print(f"  Original: {int(attn[0].sum())} tokens  →  Expanded: {valid_new} tokens")
    print(f"  ABS placeholder positions: {abs_positions}")
    # Show the layout around the insertion point
    start = max(0, int(pl[0].item()) - 2)
    end = min(valid_new, int(pl[0].item()) + n_abs + 3)
    snippet_ids = new_ids[0, start:end].tolist()
    snippet_str = []
    for tid in snippet_ids:
        if tid == placeholder_token:
            snippet_str.append("<ABS>")
        else:
            snippet_str.append(tokenizer.decode([tid]))
    print(f"  Layout around insertion: {'|'.join(snippet_str)}")
    print()

print("Tokenizer loaded, ready to inspect.\n")

In [ ]:
# ── 1. BoolQ ─────────────────────────────────────────────────────────────
# Passage can be very long → test with max_length=2560
# Response is short (just "yes"/"no" + "#### yes/no"), no CoT
# We add <abs> before the answer token id

bv = 151936  # Qwen vocab size (placeholder token = bv)

inspect_dataset(BoolQDataset, tokenizer, max_length=2560, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Also check test split size ---")
boolq_test = BoolQDataset(split="test", tokenizer=tokenizer, max_length=2560)
print(f"  BoolQ test: {len(boolq_test)} samples")
boolq_train = BoolQDataset(split="train", tokenizer=tokenizer, max_length=2560)
print(f"  BoolQ train: {len(boolq_train)} samples")

In [ ]:
# ── 2. OpenBookQA ────────────────────────────────────────────────────────
# query < 330 tok, response < 150 tok → max_length=768 should be safe
# 4-way multiple choice

inspect_dataset(OpenBookQADataset, tokenizer, max_length=768, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Split sizes ---")
obqa_train = OpenBookQADataset(split="train", tokenizer=tokenizer, max_length=768)
obqa_val   = OpenBookQADataset(split="validation", tokenizer=tokenizer, max_length=768)
obqa_test  = OpenBookQADataset(split="test", tokenizer=tokenizer, max_length=768)
print(f"  OpenBookQA train: {len(obqa_train)}, val: {len(obqa_val)}, test: {len(obqa_test)}")

In [ ]:
# ── 3. AQuA-RAT ──────────────────────────────────────────────────────────
# Question < 400 tok, answer (with rationale) < 740 tok → max_length=1024
# 5-way multiple choice with CoT rationale

inspect_dataset(AQuADataset, tokenizer, max_length=1024, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Split sizes ---")
aqua_train = AQuADataset(split="train", tokenizer=tokenizer, max_length=1024)
aqua_test  = AQuADataset(split="test", tokenizer=tokenizer, max_length=1024)
print(f"  AQuA train: {len(aqua_train)}, test: {len(aqua_test)}")

In [ ]:
# ── 4. HotpotQA — exploratory (not yet in pt_dataset.py) ─────────────────
# Check raw data format and token lengths to see if it's feasible
from datasets import load_dataset

hpqa = load_dataset("hotpot_qa", "fullwiki", split="train", trust_remote_code=True)
print(f"HotpotQA train: {len(hpqa)} samples")

# Look at fields
print(f"\nFields: {list(hpqa.features.keys())}")

# Inspect a few samples
for i in range(3):
    ex = hpqa[i]
    q = ex["question"]
    ans = ex["answer"]
    # supporting_facts provides (title, sent_idx) pairs
    sf_titles = ex["supporting_facts"]["title"] if "supporting_facts" in ex else []
    context_titles = ex["context"]["title"] if "context" in ex else []
    context_sents = ex["context"]["sentences"] if "context" in ex else []
    
    # Build a simple context string from supporting paragraphs
    context_str = ""
    for t_idx, title in enumerate(context_titles):
        sents = context_sents[t_idx]
        context_str += f"{title}: {''.join(sents)}\n"
    
    # Tokenize to measure lengths
    q_toks = len(tokenizer(q, add_special_tokens=False)["input_ids"])
    ctx_toks = len(tokenizer(context_str, add_special_tokens=False)["input_ids"])
    ans_toks = len(tokenizer(ans, add_special_tokens=False)["input_ids"])
    
    print(f"\n--- HotpotQA sample {i} ---")
    print(f"  question ({q_toks} tok): {q}")
    print(f"  answer ({ans_toks} tok): {ans}")
    print(f"  context ({ctx_toks} tok): {context_str[:200]}...")
    print(f"  type: {ex.get('type', '?')}, level: {ex.get('level', '?')}")

# Token length distribution for first 2000
print("\n--- Token length distribution (first 2000 samples) ---")
q_lens, ctx_lens, ans_lens, total_lens = [], [], [], []
for i in range(min(2000, len(hpqa))):
    ex = hpqa[i]
    q_t = len(tokenizer(ex["question"], add_special_tokens=False)["input_ids"])
    a_t = len(tokenizer(ex["answer"], add_special_tokens=False)["input_ids"])
    ctx = ""
    for t_idx, title in enumerate(ex["context"]["title"]):
        ctx += f"{title}: {''.join(ex['context']['sentences'][t_idx])}\n"
    c_t = len(tokenizer(ctx, add_special_tokens=False)["input_ids"])
    q_lens.append(q_t)
    ctx_lens.append(c_t)
    ans_lens.append(a_t)
    total_lens.append(q_t + c_t + a_t + 10)  # +10 for formatting tokens

for name, vals in [("question", q_lens), ("context", ctx_lens),
                   ("answer", ans_lens), ("total_est", total_lens)]:
    a = np.array(vals)
    print(f"  {name:>12s}  min={a.min():5d}  median={int(np.median(a)):5d}  "
          f"mean={a.mean():7.1f}  p95={int(np.percentile(a,95)):5d}  max={a.max():5d}")

print(f"\nRecommendation: cap context or use supporting-facts-only to fit within max_length")

### Summary — recommended max_length settings

| Dataset | Train | Test | max_length | max_new_tokens | Notes |
|---------|-------|------|-----------|----------------|-------|
| BoolQ | ~9.4k | ~3.2k | 2560 | 32 | Passage very long, response very short (yes/no), no CoT |
| OpenBookQA | ~5k | ~500 | 768 | 128 | 4-way MC, short query+response |
| AQuA | ~5k | ~500 | 1024 | 768 | 5-way MC with CoT rationale, response can be long |
| HotpotQA | ~90k | ~7.4k | TBD | TBD | Multi-hop QA, context very long — may need truncation |